# 2. Основы тензоров PyTorch

**Цель:** Изучить создание тензоров, операции, типы данных, устройства и автоматическое дифференцирование.

---

In [16]:
# Импортируем библиотеки
import sys
import os
import logging

# Настройка логирования: читаем уровень из LOG_LEVEL (по умолч. DEBUG)
LOG_LEVEL = os.getenv("LOG_LEVEL", "DEBUG")
logging.basicConfig(level=getattr(logging, LOG_LEVEL), format="%(asctime)s [%(levelname)s] %(name)s: %(message)s", stream=sys.stderr)
log = logging.getLogger("tensors")

import torch
import numpy as np
log.info("PyTorch %s loaded", torch.__version__)


2026-05-31 19:19:37,811 [INFO] tensors: PyTorch 2.8.0 loaded


## 2.1 Создание тензоров

In [17]:
log.debug("Creating tensors from different sources")

# Из списка Python — torch.tensor() определяет тип по содержимому
t1 = torch.tensor([1, 2, 3, 4, 5])
print(f"From list: {t1}, shape={t1.shape}")    # shape = (5,) — одномерный тензор

# Из numpy-массива — полезно при работе с существующими данными
t2 = torch.tensor(np.array([[1, 2], [3, 4]]))
print(f"From numpy:\n{t2}")

# torch.randn — случайные числа из стандартного нормального распределения N(0,1)
# shape указывается как отдельные аргументы: randn(rows, cols)
t3 = torch.randn(2, 3)
print(f"Random normal:\n{t3}")

# Специальные тензоры: нули, единицы, единичная матрица
t4 = torch.zeros(2, 4)   # все элементы = 0
t5 = torch.ones(3, 3)    # все элементы = 1
t6 = torch.eye(4)        # единичная матрица (1 на диагонали)
print(f"Zeros:\n{t4}")
print(f"Ones:\n{t5}")
print(f"Identity:\n{t6}")

# torch.arange(start, end, step) — как range() в Python
# torch.linspace(start, end, steps) — равномерно распределённые точки
t7 = torch.arange(0, 10, 2)      # [0, 2, 4, 6, 8]
t8 = torch.linspace(0, 1, 5)     # [0.0, 0.25, 0.5, 0.75, 1.0]
print(f"Arange: {t7}")
print(f"Linspace: {t8}")


2026-05-31 19:19:42,392 [DEBUG] tensors: Creating tensors from different sources


From list: tensor([1, 2, 3, 4, 5]), shape=torch.Size([5])
From numpy:
tensor([[1, 2],
        [3, 4]])
Random normal:
tensor([[ 1.1348,  1.1746,  0.1268],
        [-1.0193,  0.7272, -0.9654]])
Zeros:
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.]])
Ones:
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
Identity:
tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])
Arange: tensor([0, 2, 4, 6, 8])
Linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


## 2.2 Типы данных (dtype) и устройства (device)

In [3]:
log.debug("Exploring dtypes and devices")

# Типы данных: по умолчанию torch.float32, но можно задать явно
# float32 — основной тип для нейросетей (компромисс точность/память)
# float64 — выше точность, вдвое больше памяти
# int64 — для индексов, меток классов
float_t = torch.tensor([1.0, 2.0], dtype=torch.float32)
double_t = torch.tensor([1.0, 2.0], dtype=torch.float64)
int_t = torch.tensor([1, 2], dtype=torch.int64)
print(f"float32: {float_t.dtype}")
print(f"float64: {double_t.dtype}")
print(f"int64: {int_t.dtype}")

# Явное приведение типа: .to() или .float(), .long(), .int()
x = torch.tensor([1, 2, 3], dtype=torch.float32)
print(f"Default: {x.dtype}")

# Перемещение на устройство (GPU/MPS): .to(device) или device= при создании
# Это критично для производительности — все тензоры должны быть на одном устройстве
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
t_gpu = torch.tensor([1, 2, 3], device=device)
print(f"On device: {t_gpu.device}")
log.info("Using device: %s", device)


2026-05-31 11:20:48,759 [DEBUG] tensors: Exploring dtypes and devices
2026-05-31 11:20:48,865 [INFO] tensors: Using device: mps


float32: torch.float32
float64: torch.float64
int64: torch.int64
Default: torch.float32
On device: mps:0


## 2.3 Индексация и срезы

In [4]:
log.debug("Indexing and slicing")

# Индексация в PyTorch работает как в NumPy
# Тензор 4×5: 4 строки, 5 столбцов
x = torch.randn(4, 5)
print(f"Tensor 4x5:\n{x}\n")

print(f"First row: {x[0]}")          # первая строка (индекс 0)
print(f"First column: {x[:, 0]}")    # первый столбец (все строки, 0-й столбец)
print(f"Submatrix (1:3, 1:4):\n{x[1:3, 1:4]}")  # срез: строки 1-2, столбцы 1-3
print(f"Element (2, 3): {x[2, 3]}")  # конкретный элемент [строка, столбец]

# Boolean indexing: маска из True/False, выбираем элементы по условию
mask = x > 0
print(f"Positive values: {x[mask]}")  # все положительные элементы
print(f"Number of positive: {mask.sum()}")  # количество True в маске


2026-05-31 11:21:27,482 [DEBUG] tensors: Indexing and slicing


Tensor 4x5:
tensor([[ 0.1125, -0.8925,  0.9349,  0.3956,  0.4005],
        [ 1.7723, -0.0158,  1.4617,  0.3904,  0.8494],
        [-0.5544, -1.2242, -0.4065, -0.7399, -0.7620],
        [ 0.3084, -1.0830,  0.4076,  0.3870, -0.1861]])

First row: tensor([ 0.1125, -0.8925,  0.9349,  0.3956,  0.4005])
First column: tensor([ 0.1125,  1.7723, -0.5544,  0.3084])
Submatrix (1:3, 1:4):
tensor([[-0.0158,  1.4617,  0.3904],
        [-1.2242, -0.4065, -0.7399]])
Element (2, 3): -0.7399434447288513
Positive values: tensor([0.1125, 0.9349, 0.3956, 0.4005, 1.7723, 1.4617, 0.3904, 0.8494, 0.3084,
        0.4076, 0.3870])
Number of positive: 11


In [5]:
log.debug("Reshape operations")

x = torch.arange(12)
print(f"Original: {x.shape} -> {x}")

# view() — возвращает новое представление (view) тех же данных
# Данные разделяются в памяти: изменение y изменит x!
# Работает только для contiguous тензоров
y = x.view(3, 4)
print(f"View (3,4):\n{y}")

# reshape() — то же что view(), но делает копию если тензор не contiguous
# contiguous = элементы расположены в памяти подряд (по строкам)
# После transpose() тензор может стать non-contiguous — view() не сработает
z = x.reshape(2, 6)
print(f"Reshape (2,6):\n{z}")

# flatten() — превращает любую форму в 1D (полезно для классификатора)
w = torch.randn(2, 3, 4)
print(f"Flatten {w.shape} -> {w.flatten().shape}")

# transpose() — меняет местами две размерности
# Внимание: transpose делает тензор non-contiguous!
t = torch.randn(2, 3)
print(f"Transpose {t.shape} -> {t.T.shape}")

# unsqueeze() — добавляет размерность (полезно для broadcasting)
# squeeze() — убирает размерности длины 1
a = torch.randn(3)
print(f"Unsqueeze {a.shape} -> {a.unsqueeze(0).shape} -> {a.unsqueeze(1).shape}")
b = torch.randn(1, 3, 1, 4)
print(f"Squeeze {b.shape} -> {b.squeeze().shape}")


2026-05-31 11:22:56,049 [DEBUG] tensors: Reshape operations


Original: torch.Size([12]) -> tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
View (3,4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
Reshape (2,6):
tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])
Flatten torch.Size([2, 3, 4]) -> torch.Size([24])
Transpose torch.Size([2, 3]) -> torch.Size([3, 2])
Unsqueeze torch.Size([3]) -> torch.Size([1, 3]) -> torch.Size([3, 1])
Squeeze torch.Size([1, 3, 1, 4]) -> torch.Size([3, 4])


In [6]:
log.debug("Reshape operations")

x = torch.arange(12)
print(f"Original: {x.shape} -> {x}")

# view (разделяет память)
y = x.view(3, 4)
print(f"View (3,4):\n{y}")

# reshape (может копировать)
z = x.reshape(2, 6)
print(f"Reshape (2,6):\n{z}")

# Flatten
w = torch.randn(2, 3, 4)
print(f"Flatten {w.shape} -> {w.flatten().shape}")

# Transpose
t = torch.randn(2, 3)
print(f"Transpose {t.shape} -> {t.T.shape}")

# Unsqueeze / squeeze
a = torch.randn(3)
print(f"Unsqueeze {a.shape} -> {a.unsqueeze(0).shape} -> {a.unsqueeze(1).shape}")
b = torch.randn(1, 3, 1, 4)
print(f"Squeeze {b.shape} -> {b.squeeze().shape}")

2026-05-31 11:22:59,142 [DEBUG] tensors: Reshape operations


Original: torch.Size([12]) -> tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])
View (3,4):
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
Reshape (2,6):
tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])
Flatten torch.Size([2, 3, 4]) -> torch.Size([24])
Transpose torch.Size([2, 3]) -> torch.Size([3, 2])
Unsqueeze torch.Size([3]) -> torch.Size([1, 3]) -> torch.Size([3, 1])
Squeeze torch.Size([1, 3, 1, 4]) -> torch.Size([3, 4])


## 2.5.5 Broadcasting: подробный разбор

Broadcasting позволяет выполнять операции над тензорами разной формы.
**Правила broadcasting (по приоритету):**

1. Если размерности различаются по длине, к меньшей добавляются единицы слева
2. Размерность совместима, если равна 1 или совпадает с другой
3. Если размерность = 1, она «растягивается» до нужного размера (без копирования памяти)

**Пример:**
- `A.shape = (3, 1)` и `B.shape = (4,)` → `B.shape → (1, 4)` → `result.shape = (3, 4)`
- Это НЕ копирование данных — PyTorch виртуально растягивает тензор на лету

**Где это используется в трансформерах:**
- Добавление bias к выходу линейного слоя: `x @ W.T + bias`
- Маски: `(batch, 1, seq_len)` + `(1, seq_len, seq_len)` = `(batch, seq_len, seq_len)`
- Нормализация: вычитание среднего `(batch, 1)` из `(batch, seq_len)`


## 2.5 Математические операции

In [7]:
log.debug("Math operations")

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print(f"a + b = {a + b}")
print(f"a - b = {a - b}")
print(f"a * b = {a * b}")  # поэлементное
print(f"a / b = {a / b}")
print(f"a ** 2 = {a ** 2}")

# Матричное умножение
A = torch.randn(3, 4)
B = torch.randn(4, 2)
C = torch.mm(A, B)  # или A @ B
C2 = A @ B
print(f"Matmul: {A.shape} @ {B.shape} = {C.shape}")

# Batch matmul
batch_A = torch.randn(8, 3, 4)
batch_B = torch.randn(8, 4, 5)
batch_C = torch.bmm(batch_A, batch_B)
print(f"Batch matmul: {batch_C.shape}")

2026-05-31 11:23:03,289 [DEBUG] tensors: Math operations


a + b = tensor([5., 7., 9.])
a - b = tensor([-3., -3., -3.])
a * b = tensor([ 4., 10., 18.])
a / b = tensor([0.2500, 0.4000, 0.5000])
a ** 2 = tensor([1., 4., 9.])
Matmul: torch.Size([3, 4]) @ torch.Size([4, 2]) = torch.Size([3, 2])
Batch matmul: torch.Size([8, 3, 5])


## 2.6 Broadcasting

In [8]:
log.debug("Broadcasting demonstration")

# Скаляр + тензор: скаляр растягивается до формы тензора
x = torch.ones(3, 4)
y = x + 5  # 5 → (1,1) → (3,4) виртуально
print(f"Tensor + scalar:\n{y}")

# Разные формы: (3,1) + (4,) → (3,4)
# Правила: 4 → (1,4), затем (3,1) + (1,4) → (3,4)
a = torch.randn(3, 1)  # столбец: 3 строки, 1 признак
b = torch.randn(4)     # строка: 4 элемента
c = a + b              # broadcasting: (3,1) + (4,) -> (3,4)
print(f"Broadcast {a.shape} + {b.shape} -> {c.shape}")

# Практика: нормализация батча вручную через broadcasting
# batch: [16 примеров, 64 признака]
# mean: среднее по 16 примерам → (1, 64), broadcasting до (16, 64)
batch = torch.randn(16, 64)
mean = batch.mean(dim=0, keepdim=True)  # (1, 64) — keepdim сохраняет размерность
std = batch.std(dim=0, keepdim=True)    # (1, 64)
normalized = (batch - mean) / std        # broadcasting: (16,64) - (1,64) / (1,64)
print(f"Manual batch norm: input {batch.shape}, output {normalized.shape}")
print(f"Normalized mean: {normalized.mean():.4f}, std: {normalized.std():.4f}")


2026-05-31 11:23:09,412 [DEBUG] tensors: Broadcasting demonstration


Tensor + scalar:
tensor([[6., 6., 6., 6.],
        [6., 6., 6., 6.],
        [6., 6., 6., 6.]])
Broadcast torch.Size([3, 1]) + torch.Size([4]) -> torch.Size([3, 4])
Manual batch norm: input torch.Size([16, 64]), output torch.Size([16, 64])
Normalized mean: 0.0000, std: 0.9687


## Размерности в нейросетях: batch, features, sequence

В нейросетях тензоры обычно имеют 3 ключевые размерности:

| Размерность | Смысл | Пример |
|-------------|-------|--------|
| **Batch** (N) | Сколько примеров обрабатывается одновременно | 32, 64, 128 |
| **Features** (d) | Размер одного признака/эмбеддинга | 512, 768 (BERT) |
| **Sequence** (n) | Длина последовательности | 50, 200, 512 |

В трансформерах тензоры обычно 3D: `(batch, seq_len, d_model)`
- `attention_scores = Q @ K.T` → `(batch, n_heads, seq_len, seq_len)`
- После `softmax`, `attention @ V` → `(batch, n_heads, seq_len, d_k)`
- Понимание размерностей — половина успеха в трансформерах!


In [9]:
log.debug("Autograd on chain rule")

# Функция двух переменных: f(x,y) = x^2 * y + y^3
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# Прямой проход: f считается, граф строится
f = x ** 2 * y + y ** 3
f.backward()  # обратный проход: df/dx, df/dy

print(f"x = {x.item()}, y = {y.item()}")
print(f"f(x,y) = x^2*y + y^3 = {f.item()}")
print(f"df/dx = 2*x*y = {x.grad.item():.2f} (expected: {2*2*3:.2f})")
print(f"df/dy = x^2 + 3*y^2 = {y.grad.item():.2f} (expected: {4 + 27:.2f})")


2026-05-31 11:23:41,451 [DEBUG] tensors: Autograd on chain rule


x = 2.0, y = 3.0
f(x,y) = x^2*y + y^3 = 39.0
df/dx = 2*x*y = 12.00 (expected: 12.00)
df/dy = x^2 + 3*y^2 = 31.00 (expected: 31.00)


## 2.7 Автоматическое дифференцирование (Autograd)

In [10]:
log.debug("Autograd basics")

# requires_grad=True — говорим PyTorch следить за операциями над этим тензором
x = torch.tensor([2.0, 3.0], requires_grad=True)
print(f"x: {x}, requires_grad: {x.requires_grad}")

# Строим граф вычислений: каждая операция сохраняется для backward
y = x ** 2 + 2 * x + 1                # y = x^2 + 2x + 1
print(f"y = x^2 + 2x + 1: {y}")

# backward() требует скаляр — суммируем y
z = y.sum()
print(f"z = sum(y): {z}")

# Обратный проход: chain rule от z к x
z.backward()
# x.grad содержит dz/dx
# Математически: y = x^2 + 2x + 1, z = sum(y), dz/dx = 2x + 2
# Для x = [2, 3]: dz/dx = [2*2+2, 2*3+2] = [6, 8]
print(f"dz/dx: {x.grad}")


2026-05-31 11:23:47,709 [DEBUG] tensors: Autograd basics


x: tensor([2., 3.], requires_grad=True), requires_grad: True
y = x^2 + 2x + 1: tensor([ 9., 16.], grad_fn=<AddBackward0>)
z = sum(y): 25.0
dz/dx: tensor([6., 8.])


In [11]:
log.debug("Autograd on chain rule")

# Функция двух переменных: f(x,y) = x^2 * y + y^3
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# Прямой проход: f считается, граф строится
f = x ** 2 * y + y ** 3
f.backward()  # обратный проход: df/dx, df/dy

print(f"x = {x.item()}, y = {y.item()}")
print(f"f(x,y) = x^2*y + y^3 = {f.item()}")
print(f"df/dx = 2*x*y = {x.grad.item():.2f} (expected: {2*2*3:.2f})")
print(f"df/dy = x^2 + 3*y^2 = {y.grad.item():.2f} (expected: {4 + 27:.2f})")


2026-05-31 11:23:52,630 [DEBUG] tensors: Autograd on chain rule


x = 2.0, y = 3.0
f(x,y) = x^2*y + y^3 = 39.0
df/dx = 2*x*y = 12.00 (expected: 12.00)
df/dy = x^2 + 3*y^2 = 31.00 (expected: 31.00)


In [12]:
log.debug("Gradient accumulation and zeroing")

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# Первый backward: loss = sum(x^2)
loss = (x ** 2).sum()
loss.backward()
print(f"After first backward: {x.grad}")  # d/dx = 2x -> [2, 4, 6]

# ВАЖНО: градиенты НАКАПЛИВАЮТСЯ!
# Если не обнулить, второй backward добавит к существующему .grad
loss2 = (x * 2).sum()
loss2.backward()
print(f"After second backward (ACCUMULATED): {x.grad}")  # [2,4,6] + [2,2,2]

# Обнуление: zero_() — in-place операция (подчёркивание в PyTorch)
x.grad.zero_()
print(f"After zero_(): {x.grad}")


2026-05-31 11:23:55,897 [DEBUG] tensors: Gradient accumulation and zeroing


After first backward: tensor([2., 4., 6.])
After second backward (ACCUMULATED): tensor([4., 6., 8.])
After zero_(): tensor([0., 0., 0.])


In [13]:
log.debug("detach() and no_grad()")

# detach() — отсоединяет тензор от графа вычислений
# Полезно: взять выход модели, передать в визуализацию без трассировки
x = torch.tensor([2.0], requires_grad=True)

# y = x^2, z = y.detach() — z не отслеживается
y = x ** 2
z = y.detach()  # z не требует градиента, отрезан от графа
w = z ** 2       # w тоже не требует (пришёл из detach)
print(f"z.requires_grad: {z.requires_grad}")

# torch.no_grad() — контекст БЕЗ построения графа
# Экономит память: не хранит промежуточные значения для backward
# Используется: evaluation, визуализация, вычисление метрик
with torch.no_grad():
    y_no_grad = x ** 3  # граф не строится
    print(f"y_no_grad.requires_grad: {y_no_grad.requires_grad}")

print("Detach and no_grad demonstrated")


2026-05-31 11:23:58,302 [DEBUG] tensors: detach() and no_grad()


z.requires_grad: False
y_no_grad.requires_grad: False
Detach and no_grad demonstrated


## 2.8 Функция потерь: MSE вручную vs torch

MSE (Mean Squared Error) — среднеквадратичная ошибка:

$$
\mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2
$$

Градиент MSE: $\frac{\partial \mathcal{L}}{\partial \hat{y}_i} = \frac{2}{N} (\hat{y}_i - y_i)$


In [14]:
log.debug("Autograd on chain rule")

# Функция двух переменных: f(x,y) = x^2 * y + y^3
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# Прямой проход: f считается, граф строится
f = x ** 2 * y + y ** 3
f.backward()  # обратный проход: df/dx, df/dy

print(f"x = {x.item()}, y = {y.item()}")
print(f"f(x,y) = x^2*y + y^3 = {f.item()}")
print(f"df/dx = 2*x*y = {x.grad.item():.2f} (expected: {2*2*3:.2f})")
print(f"df/dy = x^2 + 3*y^2 = {y.grad.item():.2f} (expected: {4 + 27:.2f})")


2026-05-31 11:24:03,102 [DEBUG] tensors: Autograd on chain rule


x = 2.0, y = 3.0
f(x,y) = x^2*y + y^3 = 39.0
df/dx = 2*x*y = 12.00 (expected: 12.00)
df/dy = x^2 + 3*y^2 = 31.00 (expected: 31.00)


## Выводы

In [15]:
print("=== Tensor fundamentals complete ===")
print("Topics covered:")
print("  - Tensor creation (from list, numpy, random, zeros, ones)")
print("  - Data types (dtype) and devices (device)")
print("  - Indexing, slicing, reshaping (view, reshape, transpose)")
print("  - Math operations and matrix multiplication")
print("  - Broadcasting")
print("  - Autograd: backward, gradients, detach, no_grad")
print("  - Loss functions and gradient computation")
log.info("Tensor fundamentals notebook complete")

2026-05-31 11:24:06,784 [INFO] tensors: Tensor fundamentals notebook complete


=== Tensor fundamentals complete ===
Topics covered:
  - Tensor creation (from list, numpy, random, zeros, ones)
  - Data types (dtype) and devices (device)
  - Indexing, slicing, reshaping (view, reshape, transpose)
  - Math operations and matrix multiplication
  - Broadcasting
  - Autograd: backward, gradients, detach, no_grad
  - Loss functions and gradient computation


📚 **Полезные ссылки:**
- [PyTorch: Tensor docs](https://pytorch.org/docs/stable/tensors.html)
- [PyTorch: Broadcasting semantics](https://pytorch.org/docs/stable/notes/broadcasting.html)
- [PyTorch: Autograd mechanics](https://pytorch.org/docs/stable/notes/autograd.html)
